In [7]:
import time

import pandas as pd
import requests
from statsmodels.tsa.statespace.sarimax import SARIMAX
from datetime import datetime

In [4]:
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

WINDOW_START = pd.Timestamp("2026-02-28 12:00:00")
WINDOW_END = pd.Timestamp("2026-03-01 12:00:00")

def _straddle(value: float) -> float:
    if value < 0.2:
        return 0.2 - value
    elif value > 0.25:
        return value - 0.25
    return 0.0

def fetch_tide_fair_value() -> float:
    """Fetch Thames tidal data, fit SARIMA, and compute TIDE_SWING fair value."""
    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": 500},
    )
    resp.raise_for_status()
    items = resp.json().get("items", [])

    df = pd.DataFrame(items)[["dateTime", "value"]].rename(
        columns={"dateTime": "time", "value": "level"}
    )
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time").reset_index(drop=True)

    df["time"] = pd.to_datetime(df["time"])
    df = df[df["time"] >= pd.to_datetime("2026-02-01 00:00:00+00:00")]
    df["time"] = df["time"].dt.tz_convert(None)
    df = df.set_index("time").sort_index().asfreq("15min")

    # Fit SARIMA
    model = SARIMAX(
        df["level"],
        order=(1, 0, 1),
        seasonal_order=(1, 0, 1, 50),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    results = model.fit(disp=False)

    # Forecast to target time
    last_time = df.index[-1]
    steps = max(1, int((WINDOW_END - last_time).total_seconds() / 900))
    forecast = results.get_forecast(steps=steps)

    # Combine in-sample + out-of-sample
    insample_pred = results.get_prediction(start=0).predicted_mean
    oos_pred = forecast.predicted_mean
    all_pred = pd.concat([insample_pred, oos_pred])

    sarima_pred_df = pd.DataFrame({"pred_mean": all_pred})
    sarima_pred_df["level"] = df["level"].reindex(sarima_pred_df.index)

    # Slice window & compute price
    window = sarima_pred_df.loc[WINDOW_START:WINDOW_END].copy()
    window["level_diff"] = window["pred_mean"].diff().abs()
    window["price"] = window["level_diff"].apply(_straddle)

    fv = window["price"].sum() * 100
    return fv

In [5]:
fetch_tide_fair_value()

np.float64(640.3532103511653)

In [ ]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278
WEATHER_REFRESH_SECS = 900  # 15 minutes

# Competition window (London time, naive datetimes to match API output)
WINDOW_START = datetime(2026, 2, 28, 12, 0)
WINDOW_END = datetime(2026, 3, 1, 12, 0)


def fetch_wx_sum(past_steps: int = 96, forecast_steps: int = 96) -> float:
    """Fetch 15-min weather for London and compute WX_SUM fair value.

    WX_SUM = sum over competition window of (temp_F * humidity / 100)
    where temp_F = temp_C * 9/5 + 32.
    """
    resp = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": LONDON_LAT,
            "longitude": LONDON_LON,
            "minutely_15": "temperature_2m,relative_humidity_2m",
            "past_minutely_15": past_steps,
            "forecast_minutely_15": forecast_steps,
            "timezone": "Europe/London",
        },
    )
    resp.raise_for_status()
    m = resp.json()["minutely_15"]

    wx_sum = 0.0
    for t_str, temp_c, hum in zip(m["time"], m["temperature_2m"], m["relative_humidity_2m"]):
        if temp_c is None or hum is None:
            continue
        dt = datetime.fromisoformat(t_str)
        if WINDOW_START <= dt < WINDOW_END:
            temp_f = temp_c * 9 / 5 + 32
            wx_sum += temp_f * hum / 100

    return wx_sum

In [9]:
fetch_wx_sum()

3233.7723999999985